In [1]:
library(tidyr)
library(dplyr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
data_full <- read.csv(file = "Raw_Temperature_Screen.csv")

In [3]:
head(data_full)

,Cell_Line,Temperature,Time,Rep,Area,Batch,Incubator
,<chr>,<dbl>,<int>,<int>,<dbl>,<int>,<chr>
1,X1690,16,0,0,0,1,B
2,X1690,16,0,1,0,1,B
3,X1690,16,0,2,0,1,B
4,X1690,16,0,3,0,1,B
5,X1690,16,0,4,0,1,B
6,X1690,16,0,5,0,1,B


In [4]:
# Turn all the 25s into a single 25, but maintaining batch and incubator differences
# Remove JG5
unique(data_full$Temperature)
data_full <- data_full %>% mutate(Temperature = case_when((Temperature >= 25 & Temperature <= 26) ~ 25, 
                                                .default = Temperature), 
                       Batch = factor(Batch), 
                       Incubator = factor(Incubator)) %>%
                filter(Cell_Line != "JG5")
unique(data_full$Temperature)

data_full %>% filter(Temperature == 25) %>% pull(Batch) %>% table()
head(data_full)

[1] 16.0 20.0 25.0 25.1 25.2 25.3 28.0 30.0 32.0 35.0 37.0 39.0

[1] 16 20 25 28 30 32 35 37 39

.
   1    2    3    4 
1440 1185 1185 1185 

,Cell_Line,Temperature,Time,Rep,Area,Batch,Incubator
,<chr>,<dbl>,<int>,<int>,<dbl>,<fct>,<fct>
1,X1690,16,0,0,0,1,B
2,X1690,16,0,1,0,1,B
3,X1690,16,0,2,0,1,B
4,X1690,16,0,3,0,1,B
5,X1690,16,0,4,0,1,B
6,X1690,16,0,5,0,1,B


In [5]:
# Fix the time 
sets <- c(2,4,5)
data_full$Day <- ifelse(data_full$Time == 0, yes = 0, no = NA)
data_full$Day <- ifelse(data_full$Time == (sets[1]+24) | data_full$Time == (sets[2]+24) | data_full$Time == (sets[3]+24), 
                             yes = 1, no = data_full$Day)
data_full$Day <- ifelse(data_full$Time == 50 | data_full$Time == 52 | data_full$Time == 53, 
                             yes = 2, no = data_full$Day)
data_full$Day <- ifelse(data_full$Time == 74 | data_full$Time == 76 | data_full$Time == 77, 
                             yes = 3, no = data_full$Day)
data_full$Day <- ifelse(data_full$Time == 98 | data_full$Time == 100 | data_full$Time == 101, 
                             yes = 4, no = data_full$Day)

data_full$Day <- factor(data_full$Day)

table(data_full$Day)


  0   1   2   3   4 
951 951 951 951 951 

In [6]:
#Change all the cell Line names
data_full$Cell_Line <- sub('X','CC',data_full$Cell_Line)
head(data_full)

,Cell_Line,Temperature,Time,Rep,Area,Batch,Incubator,Day
,<chr>,<dbl>,<int>,<int>,<dbl>,<fct>,<fct>,<fct>
1,CC1690,16,0,0,0,1,B,0
2,CC1690,16,0,1,0,1,B,0
3,CC1690,16,0,2,0,1,B,0
4,CC1690,16,0,3,0,1,B,0
5,CC1690,16,0,4,0,1,B,0
6,CC1690,16,0,5,0,1,B,0


In [7]:
cell_lines <- unique(data_full$Cell_Line)

In [8]:
write.csv(data_full, "Parsed_Temperature_Screen.csv")